In [1]:
import pandas as pd
import re

INPUT_CSV = "clip-blip-with-probability.csv"
OUTPUT_CSV = "clean-blip-clip.csv"

df = pd.read_csv(INPUT_CSV)

top_columns = [
    "top_1",
    "top_2",
    "top_3"
]

for column in top_columns:
    if column not in df.columns:
        raise ValueError(f"Kolom {column} tidak ditemukan.")

def normalize_label(text):

    if pd.isna(text):
        return ""

    text = str(text).lower().strip()

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text


PERSON_LABELS = {
    "woman",
    "man",
    "people",
    "group",
    "couple",
    "person",
    "child",
    "family",
    "two people",
    "two women",
    "girl",
    "boy",
    "little girl",
    "two men",
    "two children",
    "three women",
    "men",
    "team",
    "young boy",
    "women",
    "children",
    "large crowd",
    "crowd",
    "three men",
    "three people",
    "each other people",
    "little boy",
    "their child",
    "kids",
    "kid",
    "young girl",
    "students",
    "runners",
    "runner",
    "young men",
    "family photo session",
    "pregnant couple",
    "small child",
    "pregnant girl",
    "another man",
    "baby girl",
    "his family",
    "person s shadow",
    "soccer team",
    "young women",
    "sunset - group",
    "person s feet",
    "family photoshooth",
    "family sunset"
}


def is_person_label(label):

    label = normalize_label(label)

    return label in PERSON_LABELS


def map_subject_category(label):

    label = normalize_label(label)

    if not is_person_label(label):
        return "other"

    if label in {
        "woman", "women", "two women", "three women",
        "young women", "girl", "young girl",
        "little girl", "pregnant girl", "baby girl"
    }:
        return "female"

    if label in {
        "man", "men", "two men", "three men",
        "young men", "boy", "young boy",
        "little boy", "another man"
    }:
        return "male"

    if label in {
        "child", "children", "two children",
        "their child", "kids", "kid", "small child"
    }:
        return "children"

    if label in {
        "people", "group", "crowd", "large crowd",
        "two people", "three people",
        "each other people", "sunset - group"
    }:
        return "people/group"

    if label in {
        "couple", "pregnant couple", "family",
        "his family", "family photo session",
        "family photoshooth", "family sunset"
    }:
        return "couple/family"

    if label in {
        "students", "team", "soccer team",
        "runner", "runners"
    }:
        return "activity/group role"

    if label in {
        "person", "person s shadow", "person s feet"
    }:
        return "person"

    return "person"


for column in top_columns:

    top_number = column.replace(
        "top_",
        ""
    )

    subject_label_column = f"subject_label_top_{top_number}"
    subject_category_column = f"subject_category_top_{top_number}"
    non_subject_label_column = f"non_subject_label_top_{top_number}"

    df[subject_label_column] = df[column].apply(
        lambda value: normalize_label(value)
        if is_person_label(value)
        else "other"
    )

    df[subject_category_column] = df[column].apply(
        lambda value: map_subject_category(value)
        if is_person_label(value)
        else "other"
    )

    df[non_subject_label_column] = df[column].apply(
        lambda value: "other"
        if is_person_label(value)
        else normalize_label(value)
    )


df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

print("File berhasil disimpan:", OUTPUT_CSV)
print("Jumlah data:", len(df))
print("Jumlah kolom akhir:", len(df.columns))

display(
    df[
        [
            "FileName",
            "top_1",
            "subject_label_top_1",
            "subject_category_top_1",
            "non_subject_label_top_1",
            "top_2",
            "subject_label_top_2",
            "subject_category_top_2",
            "non_subject_label_top_2",
            "top_3",
            "subject_label_top_3",
            "subject_category_top_3",
            "non_subject_label_top_3"
        ]
    ].head(30)
)

print("Distribusi subject_category gabungan top_1, top_2, top_3:")

subject_distribution = pd.concat(
    [
        df["subject_category_top_1"],
        df["subject_category_top_2"],
        df["subject_category_top_3"]
    ]
).value_counts()

display(subject_distribution)

print("Distribusi non_subject_label gabungan top_1, top_2, top_3:")

non_subject_distribution = pd.concat(
    [
        df["non_subject_label_top_1"],
        df["non_subject_label_top_2"],
        df["non_subject_label_top_3"]
    ]
).value_counts()

display(non_subject_distribution)

File berhasil disimpan: clean-blip-clip.csv
Jumlah data: 2689
Jumlah kolom akhir: 40


,FileName,top_1,subject_label_top_1,subject_category_top_1,non_subject_label_top_1,top_2,subject_label_top_2,subject_category_top_2,non_subject_label_top_2,top_3,subject_label_top_3,subject_category_top_3,non_subject_label_top_3
0,image_000003.jpg,elephant,other,other,elephant,group,group,people/group,other,people,people,people/group,other
1,image_000011.jpg,yellow raft,other,other,yellow raft,raft,other,other,raft,water,other,other,water
2,image_000012.jpg,ferris,other,other,ferris,front,other,other,front,woman,woman,female,other
3,image_000013.jpg,beach,other,other,beach,two women,two women,female,other,water,other,other,water
4,image_000016.jpg,group,group,people/group,other,picture,other,other,picture,people,people,people/group,other
5,image_000017.jpg,waterfall,other,other,waterfall,bridge,other,other,bridge,man,man,male,other
6,image_000019.jpg,couple,couple,couple/family,other,col,other,other,col,NaN,other,other,
7,image_000022.jpg,woman,woman,female,other,ocean,other,other,ocean,white coat,other,other,white coat
8,image_000029.jpg,river,other,other,river,sun,other,other,sun,NaN,other,other,
9,image_000032.jpg,beach,other,other,beach,child,child,children,other,woman,woman,female,other


Distribusi subject_category gabungan top_1, top_2, top_3:


other                  5859
female                  866
male                    541
people/group            520
couple/family           134
person                   67
children                 65
activity/group role      15
Name: count, dtype: int64

Distribusi non_subject_label gabungan top_1, top_2, top_3:


other                2208
beach                1034
ocean                 344
rock                  227
rocks                 177
                     ... 
photoshooth             1
sangra                  1
herself                 1
large wooden moon       1
bucket                  1
Name: count, Length: 962, dtype: int64